# 0411 · Preparación para el modelado, versión 2 (OE3, OE4)

> **Versión 2 (05-11-2026, evaluación del director).** Lee la base v4 (`REVISION_2026_11/V4/`) y escribe en `V4/MODELADO/`. Cambios:
> - **C2:** julio y agosto pasan a validación (cinco bloques mensuales, abril a agosto). Ya no hay prueba final en este conjunto: la prueba independiente será septiembre y octubre (fase 4), después de registrar la decisión.
> - **C1:** solo entran guías con T0 al menos 86,8 h (p99 de T0→entrega, 0417) antes del corte de observación que fijó 0330 v4; las abiertas sin vencer salen con su conteo.
> - **A1:** las entregas sin hora se quedan en clasificación; solo pierden el valor de horas (regresión).
> - **A2:** salen las guías con entrega anterior a la admisión (sin etiqueta en 0330 v4).
> - **A6:** guarda la huella SHA-256 de `base_modelado.parquet`; 0412 a 0416 la comprueban al cargar.
> - Se retiró la unión con la promesa calendario de la base V3_3 (no entra al modelo).

**Este es el primer cuaderno del modelado.** Se corre una vez; los cuadernos de cada modelo leen lo que deja aquí.

| Cuaderno | Qué hace | Depende de |
| --- | --- | --- |
| **0411 Preparación** (este) | Datos, exclusiones, variables, historia sin futuro y pliegues | 0330 / 0160 |
| 0412 Modelos base | Referencia sin modelo y regresión lineal / logística | 0411 |
| 0413 Random Forest | Búsqueda de hiperparámetros y validación | 0411 |
| 0414 XGBoost | Búsqueda de hiperparámetros y validación | 0411 |
| 0415 LightGBM | Búsqueda de hiperparámetros y validación | 0411 |
| 0416 Comparación y prueba final | Junta los resultados, H1/H2, calibración, umbral, decisión y prueba final (una sola vez) | 0412 a 0415 |
| 0420 Interpretabilidad | Importancias y SHAP del modelo final | 0416 |

0412 a 0415 no dependen entre sí: se pueden correr en cualquier orden, o al mismo tiempo como *jobs* separados de Databricks.

**Qué deja en `V3/MODELADO/`:** `base_modelado.parquet` (guías ya filtradas y con variables derivadas), `historia_<bloque>.parquet` (historia sin futuro de cada bloque de validación), `historia_final.parquet` (historia congelada para la prueba final) y `configuracion.json` (periodos, semilla, variables y tareas que usan los demás cuadernos). Además escribe `P4b_exclusiones.csv`.

In [ ]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [ ]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

## 0. Configuración

Todo lo que comparten los cuadernos de modelado se fija aquí (periodos, semilla, población). Para cambiarlo, se cambia aquí y se vuelve a correr 0411 y los que siguen.

In [ ]:
import os, json, datetime, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings("ignore")   # E1: sin avisos que impriman rutas del espacio de trabajo

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_festivos, tabla_exclusiones, agregar_historia2, pliegues_origen_movil,
                                       guardar_preparacion, guardar_huella)

In [ ]:
# Rutas
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"                       # archivos que comparten 0411 a 0416
RUTA_GOLD_V3       = f"{CARPETA_SALIDAS}GOLD_V4_ENVIO.parquet"         # 0330 v4 (se conserva el nombre de la variable)
RUTA_GOLD_V3_CLIMA = f"{CARPETA_SALIDAS}GOLD_V4_ENVIO_CLIMA.parquet"   # 0160 v4; si no existe, no se evalúa H2
RUTA_MANIFIESTO_0330 = f"{CARPETA_SALIDAS}P3_manifiesto.json"          # de aquí sale el corte de observación
RUTA_CALENDARIO    = f"{PATH_DATABRICKS}PARAMETRICAS/DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet"
COL_FECHA_CAL, COL_FESTIVO = None, None     # None = detección automática

In [ ]:
# Periodos (C2): cinco bloques de validación; julio y agosto dejan de ser prueba final
INICIOS_VALIDACION = ["2026-04-01", "2026-05-01", "2026-06-01", "2026-07-01", "2026-08-01"]
P99_T0_A_ENTREGA_H = 86.8                     # C1: p99 de horas entre T0 y la entrega (0417, celda 9)
CORTE_OBSERVACION  = pd.Timestamp(json.load(open(RUTA_MANIFIESTO_0330))["parametros"]["CORTE_OBSERVACION"])
FIN_EVALUABLE      = CORTE_OBSERVACION - pd.Timedelta(hours=P99_T0_A_ENTREGA_H)
FIN_DESARROLLO     = str(FIN_EVALUABLE)       # fin del último bloque de validación
FIN_PRUEBA         = FIN_DESARROLLO           # sin prueba en este conjunto (la prueba es la reserva sep–oct)
SEMILLA            = 42
print(f"Corte de observación: {CORTE_OBSERVACION} · último T0 evaluable: {FIN_EVALUABLE}")

In [ ]:
# Población y variables
USAR_VARIABLES_CLIENTE   = True    # historia sin futuro por ID_CLIENTE (ya seudonimizado en la base v3)
SOLO_SLA_UNICO           = False   # True: sensibilidad con solo SLA_UNICO
EXCLUIR_ENTREGA_SIN_HORA = False   # A1: las entregas sin hora quedan en clasificación (sin valor de horas)
REAJUSTAR_CON_CLIMA      = False   # True: tercera versión de H2, con hiperparámetros buscados aparte
USAR_PROMESA_COMO_TARGET = False   # se mantiene en False: la promesa calendario no entra al modelo

## 1. Datos

### 1.1 Carga

Lee la base de 0330 (o la de 0160 con clima, si existe) y el calendario de festivos. `ID_CLIENTE` ya viene seudonimizado; si está casi vacío, se desactivan las variables de cliente.

In [ ]:
festivos = cargar_festivos(RUTA_CALENDARIO, COL_FECHA_CAL, COL_FESTIVO)
usa_clima = os.path.exists(RUTA_GOLD_V3_CLIMA)
ruta = RUTA_GOLD_V3_CLIMA if usa_clima else RUTA_GOLD_V3
todo = duckdb.sql(f"SELECT * FROM read_parquet('{ruta}')").df()
print(f"Base: {os.path.basename(ruta)} · {len(todo):,} guías · con clima: {usa_clima}")

In [ ]:
# Guías por estado del desenlace (solo conteos)
display(todo.groupby("estado_desenlace", dropna=False).size().rename("guias").to_frame())

In [ ]:
if not USAR_VARIABLES_CLIENTE:
    todo = todo.drop(columns="ID_CLIENTE")
elif todo["ID_CLIENTE"].notna().mean() < 0.5:
    print(f"AVISO: ID_CLIENTE está vacío en {todo['ID_CLIENTE'].isna().mean():.0%} de las guías. "
          "Se desactivan las variables de cliente. Para usarlas, ejecutar 0300 versión 6, que lo guarda seudonimizado.")
    USAR_VARIABLES_CLIENTE = False
    todo = todo.drop(columns="ID_CLIENTE")

In [ ]:
for c in ["t0", "FECHA_ENVIO", "fecha_entrega", "t_etiqueta_reg", "t_etiqueta_clf"]:
    todo[c] = pd.to_datetime(todo[c])
todo["entrega_sin_hora"] = todo.fecha_entrega.notna() & (todo.fecha_entrega == todo.fecha_entrega.dt.normalize())

### 1.2 Exclusiones

Cada filtro se aplica en orden y se cuenta cuántas guías saca y qué tasa de incumplimiento tenían. Nada se borra sin registro: la tabla queda en `P4b_exclusiones.csv` y es el embudo secuencial de la Tabla 5.1 (cada fila es la anterior menos las que salen).

In [ ]:
SLA_PRINCIPAL = ["SLA_UNICO"] if SOLO_SLA_UNICO else ["SLA_UNICO", "SLA_POR_REGLA"]
pasos = [("Base v4",                         pd.Series(True, index=todo.index)),
         (f"SLA en {' o '.join(SLA_PRINCIPAL)}", todo.origen_sla.isin(SLA_PRINCIPAL)),
         ("Sin devolución ni siniestro",     ~todo.estado_desenlace.isin(["DEVUELTO", "SINIESTRADO"])),
         ("Entrega no anterior a la admisión (A2)", todo.estado_desenlace != "ENTREGA_ANTES_DE_ADMISION"),
         ("No vencido antes de T0",          ~todo.vencido_antes_de_t0.fillna(False).astype(bool)),
         ("Con etiqueta al corte (no abierta sin vencer)", todo.incumplimiento_sla.notna()),
         (f"T0 al menos {P99_T0_A_ENTREGA_H} h antes del corte (C1)", todo.t0 <= FIN_EVALUABLE)]
if EXCLUIR_ENTREGA_SIN_HORA:
    pasos.append(("Entrega con hora (no 00:00:00)", ~todo.entrega_sin_hora))
if usa_clima:
    pasos.append(("Clima completo (misma cohorte para H1 y H2)", todo.CLIMA_COMPLETO.fillna(False).astype(bool)))

In [ ]:
exclusiones, mask = tabla_exclusiones(todo, pasos)
display(exclusiones)
exclusiones.to_csv(f"{CARPETA_SALIDAS}P4b_exclusiones.csv", index=False)
df = todo[mask].copy()
# A1: las entregas sin hora siguen en clasificación; solo pierden el valor de horas
s = df.entrega_sin_hora.astype(bool)
df.loc[s, ["h_ultima_milla", "t_etiqueta_reg"]] = np.nan
print(f"Entregas sin hora: {int(s.sum()):,} guías siguen en clasificación sin valor de horas")

### 1.3 Última milla negativa

Si la entrega quedó antes de la salida a zona, las horas de última milla no son confiables: el objetivo de regresión pasa a nulo (la guía sigue en clasificación).

In [ ]:
neg = df.h_ultima_milla < 0
print(f"Última milla negativa con hora: {int(neg.sum())} guías -> objetivo de regresión nulo (se auditan, no se borran)")
df.loc[neg, ["h_ultima_milla", "t_etiqueta_reg"]] = np.nan
df = df.sort_values("t0").reset_index(drop=True)

### 1.4 Variables derivadas (todas conocidas en T0)

Franja horaria y día de la semana de la salida, combinación zona×franja (para la historia), día calendario (para el bootstrap), festivo y volumen declarado (nulo si falta una dimensión, nunca 0).

In [ ]:
df["franja"] = pd.cut(df.t0.dt.hour, [-1, 5, 11, 17, 23], labels=["MADRUGADA", "MANANA", "TARDE", "NOCHE"]).astype(str)
df["dia_semana"] = df.t0.dt.dayofweek.astype(str)
df["zona_franja"] = df["categoria_zona"].astype(str) + "_" + df["franja"]
df["dia"] = df.t0.dt.date
df["es_festivo"] = df.t0.dt.normalize().isin(festivos).astype(int)
vol = df.LARGO_ENVIO * df.ANCHO_ENVIO * df.ALTO_ENVIO
df["volumen_declarado_cm3"] = vol.where(vol > 0)
if USAR_VARIABLES_CLIENTE:
    df["ID_CLIENTE"] = df["ID_CLIENTE"].astype(str)
print(f"{len(df):,} guías; T0 de {df.t0.min()} a {df.t0.max()}")

### 1.5 Variables de los modelos

- `CATEGORICAS` y `NUMERICAS`: variables internas. Las `hist_*` se calculan en la sección 2.
- `VERSIONES`: conjuntos de variables que se comparan. «internas» siempre; «internas+clima» si hay clima (H2).
- `TAREAS`: (tarea, columna objetivo, momento en que se conoce la etiqueta).

In [ ]:
CATEGORICAS = ["regional_origen", "categoria_zona", "franja", "dia_semana", "ID_TIPOGUIA", "ID_PRODUCTO"]
NUMERICAS = ["sla_h", "margen_sla_en_t0_h", "h_admision_a_t0", "piezas_declaradas", "peso_declarado_kg",
             "volumen_declarado_cm3", "es_festivo", "hist_h_zona_franja", "hist_inc_zona_franja", "n_hist"]
if USAR_VARIABLES_CLIENTE:
    NUMERICAS += ["hist_h_cliente", "hist_inc_cliente", "n_hist_cliente"]
for c in CATEGORICAS: df[c] = df[c].astype(str)
VARIABLES_CLIMA = ["PRECIP_PREV_1H", "PRECIP_MAX_PREV_1H", "LLUVIA_3H_PREV", "LLUVIA_24H_PREV", "LLUVIA_72H_PREV",
                   "TEMP_PREV_1H", "HUM_PREV_1H"]

In [ ]:
VERSIONES = {"internas": NUMERICAS}
if usa_clima:
    assert set(VARIABLES_CLIMA) <= set(df.columns), "La base con clima no trae las columnas esperadas"
    VERSIONES["internas+clima"] = NUMERICAS + VARIABLES_CLIMA
    if REAJUSTAR_CON_CLIMA: VERSIONES["internas+clima_reajustada"] = NUMERICAS + VARIABLES_CLIMA
print("Versiones:", list(VERSIONES))
# v3.3: target clf conmutable. incumplimiento_promesa = 1 - cumplimiento_promesa (regla calendario, 0330 v3.3),
# con el mismo sentido que incumplimiento_sla (1 = tarde). t_etiqueta_promesa = cuándo quedó conocida la etiqueta.
TAREA_CLF = (("clf", "incumplimiento_promesa", "t_etiqueta_promesa") if USAR_PROMESA_COMO_TARGET
             else ("clf", "incumplimiento_sla", "t_etiqueta_clf"))
TAREAS = [TAREA_CLF, ("reg", "h_ultima_milla", "t_etiqueta_reg")]
print("Target de clasificación:", TAREA_CLF[1])

## 2. Historia sin futuro y pliegues temporales

### 2.1 Historia sin futuro

Para cada guía se calcula la media histórica de horas y de incumplimiento de su zona×franja y de su cliente, usando solo guías cuyo desenlace se conocía antes del inicio del bloque (función del asesor `media_historica_sin_futuro`). Se calcula una vez por bloque.

In [ ]:
HIST = {pd.Timestamp(ini): agregar_historia2(df, ini, USAR_VARIABLES_CLIENTE) for ini in INICIOS_VALIDACION}

### 2.2 Pliegues de origen móvil

Cada bloque de validación es un mes (abril a agosto; agosto llega hasta `FIN_EVALUABLE`). Se entrena con todo lo anterior cuya etiqueta ya se conocía al inicio del mes y se valida con ese mes. La prueba final no se toca. Aquí solo se muestran los tamaños; cada cuaderno de modelo vuelve a armar los pliegues (es instantáneo).

In [ ]:
PLIEGUES = {t: list(pliegues_origen_movil(df, INICIOS_VALIDACION, FIN_DESARROLLO, t_etiqueta=te)) for t, _, te in TAREAS}
for t, y_col, _ in TAREAS:
    for ini, tr, va in PLIEGUES[t]:
        print(f"{t} · bloque {ini.date()}: entrenamiento {df[y_col].iloc[tr].notna().sum():,}, validación {df[y_col].iloc[va].notna().sum():,}")

### 2.3 Historia congelada al final del desarrollo

La misma historia, con todo lo conocido antes de `FIN_DESARROLLO`. Queda guardada para la fase 4 (reserva de septiembre y octubre); en esta versión no hay prueba.

In [ ]:
HIST_FINAL = agregar_historia2(df, FIN_DESARROLLO, USAR_VARIABLES_CLIENTE)

## 3. Guardar para los cuadernos de modelos

In [ ]:
CONFIGURACION = {
    "INICIOS_VALIDACION": INICIOS_VALIDACION, "FIN_DESARROLLO": FIN_DESARROLLO, "FIN_PRUEBA": FIN_PRUEBA, "SEMILLA": SEMILLA,
    "USAR_VARIABLES_CLIENTE": USAR_VARIABLES_CLIENTE, "SOLO_SLA_UNICO": SOLO_SLA_UNICO,
    "EXCLUIR_ENTREGA_SIN_HORA": EXCLUIR_ENTREGA_SIN_HORA, "REAJUSTAR_CON_CLIMA": REAJUSTAR_CON_CLIMA,
    "USAR_PROMESA_COMO_TARGET": USAR_PROMESA_COMO_TARGET,
    "CORTE_OBSERVACION": str(CORTE_OBSERVACION), "P99_T0_A_ENTREGA_H": P99_T0_A_ENTREGA_H, "FIN_EVALUABLE": str(FIN_EVALUABLE),
    "CATEGORICAS": CATEGORICAS, "VERSIONES": VERSIONES, "TAREAS": TAREAS,
    "usa_clima": usa_clima, "ruta_gold": ruta, "RUTA_CALENDARIO": RUTA_CALENDARIO,
    "exclusiones": exclusiones.to_dict("records"),
    "fecha_preparacion": datetime.datetime.now().isoformat()}
guardar_preparacion(CARPETA_MODELADO, df, HIST, HIST_FINAL, CONFIGURACION)
HUELLA = guardar_huella(CARPETA_MODELADO)      # A6: 0412 a 0416 la comprueban al cargar
print("Huella SHA-256 de base_modelado.parquet:", HUELLA)
print("Preparación guardada en", CARPETA_MODELADO)
print(json.dumps({k: v for k, v in CONFIGURACION.items() if k not in ("VERSIONES", "exclusiones")}, indent=1, default=str))

In [ ]:
import os, glob, zipfile
V4 = "/Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_11/V4/"
archivos = [f for f in glob.glob(V4 + "**/*", recursive=True)
            if f.endswith((".csv", ".json")) and "/MODELADO/" not in f]
local = "/tmp/resultados_v4_0711.zip"
with zipfile.ZipFile(local, "w", zipfile.ZIP_DEFLATED) as z:
    for f in archivos:
        z.write(f, f.replace(V4, ""))
import shutil
shutil.copy(local, V4 + "resultados_v4_0711.zip")
print(len(archivos), "archivos en", V4 + "resultados_v4_0711.zip")